# 12a — Download all cutouts for Gaia-stable diaObjects

## Purpose

Download and cache on disk the three cutout stamps (science, template, difference)
for **every diaSource** belonging to the Gaia-stable diaObjects selected in
`09b_psfFlux_scienceFlux_templateFlux_GaiaDR3matching.ipynb`.

The hypothesis under investigation is that **dipole residuals** in the
PSF-subtracted difference images are responsible for triggering Rubin AP alerts
on otherwise photometrically stable Gaia stars — sources that should never appear
as transients.

## Strategy

Fink alert packets (stored as parquet in `data_FINK_BLOCK_LC_01/`) already
contain the three cutout stamps as FITS-compressed byte blobs in columns:
- `r:cutoutScience`    — 2-D science image stamp (calexp)
- `r:cutoutTemplate`  — 2-D template image stamp (coadd)
- `r:cutoutDifference`— 2-D difference image stamp (DIA)

Each stamp is a **gzip-compressed FITS file** embedded as a byte string.
We decompress and save each stamp as an individual `.fits.gz` file under
`data_cutouts_12a/<diaObjectId>/<diaSourceId>_<type>.fits.gz`.

In addition, a metadata CSV index `data_cutouts_12a/cutout_index.csv` is produced
with one row per stamp, recording diaObjectId, diaSourceId, visit, detector,
band, MJD, psfFlux, scienceFlux, templateFlux, isDipole, and the local file path.

## Inputs

| File | Produced by |
|------|-------------|
| `data_FINK_BLOCK_LC_01/gaia_star_stable_hq_src.parquet`                    | notebook 01 |
| `data_FINK_BLOCK_LC_01/gaia_nophotgstar_stable_unknown_parallax_src.parquet` | notebook 01 |
| `data_FINK_BLOCK_LC_08b/fink_diaobj_gaia_join_matched.csv`                 | notebook 08b |

## Output

```
data_cutouts_12a/
    cutout_index.csv                          ← master index
    <diaObjectId>/
        <diaSourceId>_science.fits.gz
        <diaSourceId>_template.fits.gz
        <diaSourceId>_difference.fits.gz
```

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Follows:** `09b_psfFlux_scienceFlux_templateFlux_GaiaDR3matching.ipynb`
- **Creation date:** 2026-05-13
- **Subject:** Fink/LSST DIA — Dipole hypothesis — cutout stamp download


## 1. Imports & configuration

In [ ]:
import os
import io
import gzip
import glob
import warnings

import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.time import Time
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

In [ ]:
# ── Notebook configuration ────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_12a"

# Input directories
DIR_DATA_01 = "data_FINK_BLOCK_LC_01"  # src parquets
DIR_DATA_08b = "data_FINK_BLOCK_LC_08b"  # Gaia match catalogue

# Output directory for cutout stamps
DIR_CUTOUTS = "data_cutouts_12a"
os.makedirs(DIR_CUTOUTS, exist_ok=True)

# Gaia match file (from notebook 08b)
FILE_GAIA_MATCHED = os.path.join(DIR_DATA_08b, "fink_diaobj_gaia_join_matched.csv")

# Categories of interest: only the two stable Gaia categories
# (gaia_star_variable is excluded — it is the variable control group)
STABLE_CATEGORIES = [
    "gaia_star_stable_hq",
    "gaia_nophotgstar_stable_unknown_parallax",
]

# ── Column names (Fink alert schema) ─────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_SRC = "r:diaSourceId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_VISIT = "r:visit"
COL_DET = "r:detector"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"
COL_DIPOLE = "r:isDipole"

# Science / template flux column candidates
SCIENCE_FLUX_CANDIDATES = ["r:scienceFlux", "r:psfScience", "r:psfScienceFlux"]
TEMPLATE_FLUX_CANDIDATES = ["psfTemplateFlux", "r:templateFlux", "r:template", "templateFlux"]

# Cutout column names in the Fink parquet
COL_CUT_SCI = "r:cutoutScience"
COL_CUT_TEMP = "r:cutoutTemplate"
COL_CUT_DIFF = "r:cutoutDifference"

# AB magnitude calibration
AB_FLUX_ZERO = 3631e9  # nJy → AB: m = -2.5 * log10(f / AB_FLUX_ZERO)

print(f"Cutout output dir : {os.path.abspath(DIR_CUTOUTS)}")
print(f"Stable categories : {STABLE_CATEGORIES}")

## 2. Utility functions

In [ ]:
def flux_to_mag_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude. Returns NaN for non-positive flux."""
    f = float(flux_nJy)
    if np.isfinite(f) and f > 0:
        return -2.5 * np.log10(f / AB_FLUX_ZERO)
    return np.nan


def resolve_column(df, candidates):
    """Return the first column name from *candidates* that exists in df, or None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None


def parse_dipole_flag(val):
    """Coerce a dipole flag (bool / int / str / NaN) to a Python bool."""
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)):
        return bool(val) if np.isfinite(float(val)) else False
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


def decode_cutout_bytes(raw):
    """
    Decode a raw cutout value from the Fink parquet.

    Fink stores cutouts as gzip-compressed FITS bytes in a bytes object
    (or a dict with key 'stampData').  Returns the raw bytes or None.

    Parameters
    ----------
    raw : bytes | dict | None

    Returns
    -------
    bytes or None
    """
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    if isinstance(raw, dict):
        # Fink AVRO: {'stampData': b'...'}
        raw = raw.get("stampData", raw.get("bytes", None))
    if isinstance(raw, (bytes, bytearray)):
        return bytes(raw)
    return None


def save_cutout_stamp(raw_bytes, filepath):
    """
    Save a cutout stamp to *filepath*.

    The stamp is already gzip-compressed FITS from Fink; we write it
    directly as-is (keeping the .fits.gz extension).
    If the data is not gzip-compressed, we compress it on the fly.

    Returns True on success, False on failure.
    """
    if raw_bytes is None:
        return False
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    # Detect gzip magic bytes
    is_gzip = raw_bytes[:2] == b"\x1f\x8b"
    try:
        if is_gzip:
            with open(filepath, "wb") as f:
                f.write(raw_bytes)
        else:
            # Wrap in gzip
            with gzip.open(filepath, "wb") as f:
                f.write(raw_bytes)
        return True
    except Exception as e:
        print(f"    WARNING: could not save {filepath}: {e}")
        return False


def read_cutout_as_array(filepath):
    """
    Read a saved .fits.gz cutout stamp and return the 2-D numpy array.
    Returns None if the file does not exist or cannot be parsed.
    """
    if not os.path.exists(filepath):
        return None
    try:
        with gzip.open(filepath, "rb") as f:
            hdul = fits.open(io.BytesIO(f.read()))
        # First image extension with data
        for hdu in hdul:
            if hdu.data is not None and hdu.data.ndim >= 2:
                return hdu.data.astype(float)
        return None
    except Exception:
        return None


print("Utility functions defined.")

## 3. Load Gaia DR3 matched catalogue

In [ ]:
if not os.path.exists(FILE_GAIA_MATCHED):
    raise FileNotFoundError(
        f"{FILE_GAIA_MATCHED} not found.\nRun notebook 08b_matchFinkAlertswithGaiaDR3.ipynb first."
    )

df_gaia = pd.read_csv(FILE_GAIA_MATCHED)
print(f"Gaia matched catalogue: {df_gaia.shape[0]} rows × {df_gaia.shape[1]} columns")
print(f"Columns: {df_gaia.columns.tolist()}")

# Build the set of Gaia-matched diaObjectIds
gaia_matched_ids = set(df_gaia["diaObjectId"].astype(str).values)
print(f"Unique Gaia-matched diaObjectIds: {len(gaia_matched_ids)}")

# Index for metadata lookups
df_gaia_idx = df_gaia.set_index("diaObjectId")

## 4. Load src parquets for stable Gaia categories

We load **only** the two stable-star categories; the variable-star control group
is excluded because we are specifically investigating spurious dipole alerts on
stable stars.

In [ ]:
frames = []
for cat in STABLE_CATEGORIES:
    fpath = os.path.join(DIR_DATA_01, f"{cat}_src.parquet")
    if not os.path.exists(fpath):
        print(f"[SKIP] {cat}: {fpath} not found")
        continue
    df_tmp = pd.read_parquet(fpath)
    df_tmp["gaia_origin"] = cat
    print(f"  Loaded {cat}: {len(df_tmp):,} rows")
    frames.append(df_tmp)

if not frames:
    raise FileNotFoundError(
        f"No stable-category parquet files found in {DIR_DATA_01}. Re-run notebook 01 to regenerate."
    )

df_src = pd.concat(frames, ignore_index=True)
print(f"\nCombined src shape : {df_src.shape[0]:,} rows × {df_src.shape[1]} columns")

# Detect science / template flux columns
COL_SCI = resolve_column(df_src, SCIENCE_FLUX_CANDIDATES)
COL_TEMP = resolve_column(df_src, TEMPLATE_FLUX_CANDIDATES)
print(f"scienceFlux  column : {COL_SCI}")
print(f"templateFlux column : {COL_TEMP}")

# Check cutout columns
for col in (COL_CUT_SCI, COL_CUT_TEMP, COL_CUT_DIFF):
    status = "FOUND" if col in df_src.columns else "MISSING"
    print(f"  {col:30s} : {status}")

## 5. Filter to Gaia-matched stable objects only

In [ ]:
df_src["_oid_str"] = df_src[COL_OBJ].astype(str)
n_before = len(df_src)
df_stable = df_src[df_src["_oid_str"].isin(gaia_matched_ids)].copy()
n_after = len(df_stable)

print(f"Before Gaia filter : {n_before:,} rows")
print(f"After  Gaia filter : {n_after:,} rows ({100 * n_after / n_before:.1f}%)")
print(f"Unique diaObjects  : {df_stable[COL_OBJ].nunique()}")
print()
print("Breakdown by gaia_origin:")
print(df_stable["gaia_origin"].value_counts().to_string())

## 6. Download & save cutout stamps

For each diaSource in the filtered table we extract the three stamp columns,
decode the bytes, and write them to
`data_cutouts_12a/<diaObjectId>/<diaSourceId>_<type>.fits.gz`.

Already-saved files are skipped (idempotent re-runs).

In [ ]:
# ── Check whether cutout columns are present ──────────────────────────────────
has_sci_cut = COL_CUT_SCI in df_stable.columns
has_temp_cut = COL_CUT_TEMP in df_stable.columns
has_diff_cut = COL_CUT_DIFF in df_stable.columns

if not (has_sci_cut or has_temp_cut or has_diff_cut):
    print(
        "WARNING: No cutout columns found in the parquet files.\n"
        "The Fink parquet files may have been generated without the stamp columns.\n"
        "Please check whether the Fink API download included cutouts "
        "(Fink conesearch with withcutouts=True or equivalent).\n"
        "Notebook 12b will still work if you supply cutout data via another route."
    )
else:
    print(f"Cutout columns found:")
    print(f"  {COL_CUT_SCI}    : {has_sci_cut}")
    print(f"  {COL_CUT_TEMP}   : {has_temp_cut}")
    print(f"  {COL_CUT_DIFF}   : {has_diff_cut}")

In [ ]:
index_rows = []  # will become cutout_index.csv
n_saved = {"science": 0, "template": 0, "difference": 0}
n_missing = {"science": 0, "template": 0, "difference": 0}
n_skipped = 0  # already-saved files

for row in tqdm(df_stable.itertuples(index=False), total=len(df_stable), desc="Saving cutouts"):
    obj_id = getattr(row, COL_OBJ.replace(":", "_"), None)  # tqdm tuple attribute
    # Fallback: iterate column-by-column
    rowdict = row._asdict() if hasattr(row, "_asdict") else {}

    obj_id = str(rowdict.get(COL_OBJ, ""))
    src_id = str(rowdict.get(COL_SRC, ""))
    visit = rowdict.get(COL_VISIT, np.nan)
    detector = rowdict.get(COL_DET, np.nan)
    band = rowdict.get(COL_BAND, "?")
    mjd = rowdict.get(COL_MJD, np.nan)
    psf_flux = rowdict.get(COL_PSF, np.nan)
    sci_flux = rowdict.get(COL_SCI, np.nan) if COL_SCI else np.nan
    tmp_flux = rowdict.get(COL_TEMP, np.nan) if COL_TEMP else np.nan
    dipole = parse_dipole_flag(rowdict.get(COL_DIPOLE, False))
    origin = rowdict.get("gaia_origin", "?")

    obj_dir = os.path.join(DIR_CUTOUTS, obj_id)

    stamp_paths = {}
    for stamp_type, col_cut in [
        ("science", COL_CUT_SCI),
        ("template", COL_CUT_TEMP),
        ("difference", COL_CUT_DIFF),
    ]:
        fpath = os.path.join(obj_dir, f"{src_id}_{stamp_type}.fits.gz")
        stamp_paths[stamp_type] = fpath

        if os.path.exists(fpath):
            n_skipped += 1
            continue

        raw = rowdict.get(col_cut, None) if col_cut in df_stable.columns else None
        raw_bytes = decode_cutout_bytes(raw)

        if raw_bytes is not None:
            ok = save_cutout_stamp(raw_bytes, fpath)
            if ok:
                n_saved[stamp_type] += 1
            else:
                n_missing[stamp_type] += 1
        else:
            n_missing[stamp_type] += 1

    index_rows.append(
        {
            "diaObjectId": obj_id,
            "diaSourceId": src_id,
            "visit": visit,
            "detector": detector,
            "band": band,
            "mjd": mjd,
            "psfFlux_nJy": psf_flux,
            "scienceFlux_nJy": sci_flux,
            "templateFlux_nJy": tmp_flux,
            "isDipole": dipole,
            "gaia_origin": origin,
            "path_science": stamp_paths.get("science", ""),
            "path_template": stamp_paths.get("template", ""),
            "path_difference": stamp_paths.get("difference", ""),
        }
    )

print(f"\nStamps saved   : {n_saved}")
print(f"Stamps missing : {n_missing}")
print(f"Files skipped (already existed) : {n_skipped}")

## 7. Build and save the cutout index

In [ ]:
df_index = pd.DataFrame(index_rows)

# Add AB magnitudes
df_index["mag_AB_science"] = df_index["scienceFlux_nJy"].apply(flux_to_mag_AB)
df_index["mag_AB_template"] = df_index["templateFlux_nJy"].apply(flux_to_mag_AB)
df_index["mag_AB_psf"] = df_index["psfFlux_nJy"].apply(
    lambda f: flux_to_mag_AB(abs(f)) if np.isfinite(float(f)) and f != 0 else np.nan
)


# Attach Gaia metadata from 08b catalogue
def _get_gaia_meta(oid_str):
    try:
        key = int(oid_str)
        row = df_gaia_idx.loc[key]
        return (
            row.get("gaia_phot_g_mean_mag", np.nan),
            row.get("gaia_status", "?"),
            row.get("group", "?"),
        )
    except (KeyError, ValueError):
        return np.nan, "?", "?"


meta_tuples = df_index["diaObjectId"].apply(_get_gaia_meta)
df_index["gaia_G_mag"] = [t[0] for t in meta_tuples]
df_index["gaia_status"] = [t[1] for t in meta_tuples]
df_index["fink_group"] = [t[2] for t in meta_tuples]


# ISO date string for MJD
def mjd_to_isot(mjd):
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except Exception:
        return "?"


df_index["obs_date"] = df_index["mjd"].apply(mjd_to_isot)

print(f"Index shape : {df_index.shape}")
df_index.head(3)

In [ ]:
index_path = os.path.join(DIR_CUTOUTS, "cutout_index.csv")
df_index.to_csv(index_path, index=False)
print(f"Cutout index saved → {os.path.abspath(index_path)}")
print(f"Total entries : {len(df_index):,}")

## 8. Summary statistics

In [ ]:
print("=" * 60)
print("CUTOUT DOWNLOAD SUMMARY")
print("=" * 60)
print(f"Total diaSource rows processed : {len(df_index):,}")
print(f"Unique diaObjects              : {df_index['diaObjectId'].nunique()}")
print()
print("Per gaia_origin:")
print(df_index.groupby("gaia_origin")[["diaObjectId", "diaSourceId"]].nunique().to_string())
print()
print("Per band:")
print(df_index["band"].value_counts().sort_index().to_string())
print()
print("Dipole fraction:")
n_dip = df_index["isDipole"].sum()
n_tot = len(df_index)
print(f"  isDipole=True : {n_dip:,} / {n_tot:,}  ({100 * n_dip / n_tot:.1f}%)")
print()

# Check file sizes on disk
saved_files = glob.glob(os.path.join(DIR_CUTOUTS, "**", "*.fits.gz"), recursive=True)
total_size_mb = sum(os.path.getsize(f) for f in saved_files) / 1e6
print(f"Files written to disk : {len(saved_files):,}")
print(f"Total disk size       : {total_size_mb:.1f} MB")
print(f"Output directory      : {os.path.abspath(DIR_CUTOUTS)}")

## 9. Quick sanity check — display one triplet of stamps

Pick the first available diaSource with all three stamps and display them
as a quick visual sanity check.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

try:
    import ipympl

    %matplotlib widget
except ImportError:
    %matplotlib inline


def _zscale(arr):
    """Simple percentile-based stretch for display."""
    vmin = np.nanpercentile(arr, 1)
    vmax = np.nanpercentile(arr, 99)
    return vmin, vmax


# Find the first row with all three stamps present on disk
mask = (
    df_index["path_science"].apply(os.path.exists)
    & df_index["path_template"].apply(os.path.exists)
    & df_index["path_difference"].apply(os.path.exists)
)

if mask.any():
    sample = df_index[mask].iloc[0]
    sci_arr = read_cutout_as_array(sample["path_science"])
    tmp_arr = read_cutout_as_array(sample["path_template"])
    diff_arr = read_cutout_as_array(sample["path_difference"])

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles = ["Science", "Template", "Difference"]
    arrays = [sci_arr, tmp_arr, diff_arr]

    for ax, arr, title in zip(axes, arrays, titles):
        if arr is not None:
            vmin, vmax = _zscale(arr)
            ax.imshow(arr, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
        else:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    fig.suptitle(
        f"Sanity check — diaObjectId={sample['diaObjectId']}  "
        f"diaSourceId={sample['diaSourceId']}\n"
        f"visit={sample['visit']}  det={sample['detector']}  "
        f"band={sample['band']}  isDipole={sample['isDipole']}",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()
else:
    print("No complete triplets found on disk yet.")
    print("This is expected if the parquet files lack cutout columns.")
    print("Check the output of Section 6 (n_missing) for details.")